# Fase 1 — Treino e serialização do modelo (MNIST → MLP)

Este notebook prepara o modelo que a API vai servir. Estrutura:

1. Objetivo e origem do modelo
2. Carga dos dados
3. Divisão estratificada em treino, validação e teste
4. Normalização dos pixels
5. Treino do MLP
6. Avaliação no conjunto de teste
7. Serialização com joblib
8. Relatório de métricas



## 1. Objetivo e origem do modelo

Este notebook treina o modelo que será servido pela API e grava os arquivos que ela precisa: o modelo serializado em `models/mnist_mlp.joblib` e o relatório de métricas em `reports/metrics.json`.

O modelo é um MLP (Perceptron Multicamadas), uma rede neural, reaproveitado do projeto mnist-modulo2 (classificação de dígitos MNIST). Foi escolhido porque, naquele projeto, teve uma acurácia de cerca de 97,8% no teste, a maior entre os 3 modelos comparados, a predição mais rápida: ~0,08 s para 14.000 imagens (Random Forest 0,87 s e KNN 15,3 s) e gera um arquivo pequeno.

Hiperparâmetros usados (já ajustados e validados no projeto anterior):

- `hidden_layer_sizes=(128, 64)`: duas camadas ocultas, com 128 e 64 neurônios. Controla a capacidade da rede de aprender padrões;
- `alpha=0.001`: força da regularização L2, uma “punição” a pesos muito grandes que reduz o overfitting;
- `early_stopping=True`: a rede separa parte do treino para se auto-avaliar e para quando deixa de melhorar;
- `random_state=42`: fixa a semente aleatória, para o resultado ser reproduzível.

### 1.1 Importações

In [1]:
# Importando as bibliotecas necessárias:

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib
from pathlib import Path
import json
import sklearn


## 2. Carga dos dados

O `mnist_784` tem 70000 imagens de dígitos manuscritos, cada uma com 784 pixels (imagens de 28 × 28 achatadas em uma linha). Cada coluna é a intensidade de um pixel numa posição fixa da imagem e vai de 0 (preto) a 255 (branco).

Os rótulos (`y`) chegam como texto, porque o OpenML guarda o alvo como categoria. Por isso converti para inteiro (uint8) com `astype`.


In [2]:
# Carregando o MNIST (as_frame=False), convertendo y para inteiro e imprimindo os formatos e os tipos
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, y = mnist.data, mnist.target
y = y.astype(np.uint8)  
print(X.shape, y.shape, X.dtype, X.min(), X.max())


(70000, 784) (70000,) int64 0 255



## 3. Divisão estratificada em treino, validação e teste

Dividi os dados em 70% treino, 10% validação, 20% teste. A divisão foi feita em 2 etapas: primeiro separei 20% para o teste e, dos 80% restantes, separei uma fração de 0,125 para a validação. Essa segunda fração é diferente da proporção final desejada porque é calculada sobre os 80% (10% do total ÷ 80% = 0,125.)

Usei `stratify` para manter a mesma proporção de cada dígito em todos os conjuntos. Isso importa neste dataset porque evita que, por azar do sorteio, um dígito fique sub-representado no treino ou no teste.

- **Treino:** onde o modelo aprende os padrões;
- **Validação:** ajusta e compara configurações sem gastar o teste (usada no projeto anterior)
- **Teste:** é a medida final e honesta do desempenho; só será usado no final, na avaliação (seção 6)

Fixei `random_state=42`, para a divisão ser reproduzível (mesma partição para quem rodar)

Resultado: 49000 amostras de treino, 7000 de validação e 14000 de teste.

In [3]:

from sklearn.model_selection import train_test_split


X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, stratify=y_temp, random_state=42
)

print("Treino:   ", X_train.shape, y_train.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:    ", X_test.shape, y_test.shape)

Treino:    (49000, 784) (49000,)
Validação: (7000, 784) (7000,)
Teste:     (14000, 784) (14000,)


## 4. Normalização dos pixels

Os pixels vão de 0 a 255 e eu os levei para a escala de 0.0 a 1.0, dividindo por 255.0.

Normalizar ajuda o MLP porque valores pequenos e na mesma escala deixam a otimização por gradiente mais estável e rápida; valores grandes a deixam lenta e instável.

Dividir por uma constante não vaza informação do teste para o treino porque 255 é o valor máximo teórico de um pixel de 8 bits, conhecido de antemão, e não é calculado a partir dos dados. Já um scaler ajustado com `.fit()` calcula estatísticas dos dados (média, desvio), então precisaria ser ajustado só no treino e depois aplicado nos outros conjuntos.

**Contrato com a API:** a API precisa dividir os pixels por 255.0 antes de prever, exatamente como no treino. Se isso não acontecer, o modelo recebe valores numa escala diferente da do treino e as previsões ficam erradas (isso se chama train/serve skew). Esse valor será registrado em `reports/metrics.json` (na seção 8).

In [4]:
# Normalização: divide cada pixel por 255.0 para que os valores fiquem entre 0 e 1.
X_train = X_train / 255.0
X_val   = X_val / 255.0
X_test  = X_test / 255.0
print("Min:", X_train.min(), "| Max:", X_train.max())
print("Min:", X_val.min(), "| Max:", X_val.max())
print("Min:", X_test.min(), "| Max:", X_test.max())


Min: 0.0 | Max: 1.0
Min: 0.0 | Max: 1.0
Min: 0.0 | Max: 1.0


## 5. Treino do MLP

Treinei o MLP com os hiperparâmetros da seção 1, usando apenas o conjunto de treino (X_train e y_train). Não ajustei hiperparâmetros neste notebook porque os hiperparâmetros já foram escolhidos e validados no projeto anterior.

Durante o `fit`, a rede ajusta os pesos para reduzir o erro (backpropagation). Com `early_stopping=True`, ela separa 10% do treino para se auto-avaliar e o treino para sozinho. Por isso o conjunto de validação (`X_val`) não é usado no `fit`: o `early_stopping` usa uma parte interna do treino para se auto-avaliar, então o `X_val` fica reservado.

In [5]:
modelo_mlp = MLPClassifier(hidden_layer_sizes=(128, 64), alpha=0.001, early_stopping=True, random_state=42)
modelo_mlp.fit(X_train, y_train)
print("MLP final treinado.")

MLP final treinado.


## 6. Avaliação no conjunto de teste

Avaliei o modelo uma única vez, no conjunto de teste, porque ele só é usado no final, para uma medida honesta, e não pode influenciar nenhuma escolha do modelo. Calculei três métricas:

- **Acurácia (0,9780):**  a proporção de imagens classificadas corretamente.
- **F1 ponderado (0,9780):**  o equilíbrio entre precisão e recall; “ponderado” é a média dos 10 dígitos, dando mais peso aos que têm mais imagens.
- **ROC-AUC multiclasse (0,9995):**  para cada dígito contra os outros, o quanto o modelo dá nota maior aos exemplos certos do que aos errados (usa as probabilidades), e depois tira a média.

O resultado mostra que o modelo acerta cerca de 97,8% das imagens de teste e ordena muito bem as probabilidades (AUC perto de 1), igual ao projeto anterior.

In [6]:
y_previsto=modelo_mlp.predict(X_test)
y_proba=modelo_mlp.predict_proba(X_test)
accuracy = accuracy_score(y_test, y_previsto)
f1 = f1_score(y_test, y_previsto, average='weighted')
auc = roc_auc_score(y_test, y_proba, multi_class="ovr")
print(f"Acurácia: {accuracy:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")

Acurácia: 0.9780
F1-Score: 0.9780
ROC-AUC: 0.9995


## 7. Serialização com joblib

Gravei o modelo treinado em `models/mnist_mlp.joblib` usando `joblib.dump`. O arquivo fica na pasta `models/`, na raiz do projeto, e o caminho usado é relativo porque um caminho absoluto (`C:\Users\...`) só existe na minha máquina e impediria o avaliador de rodar o código.

A API vai carregar (ler) o arquivo para a memória, uma vez, na inicialização, em vez de treinar o modelo de novo.

Para ter certeza de que o arquivo está correto, recarreguei o modelo com `joblib.load` e comparei as previsões dele com as do modelo original sobre o conjunto de teste. O resultado foi True, o que significa que todas as previsões são idênticas, ou seja, o arquivo guarda exatamente o modelo treinado.

O arquivo tem cerca de 2572 KB e só deve ser carregado com a mesma versão do scikit-learn usada no treino (1.9.1), por isso as versões estão fixadas no `requirements.txt`.


In [7]:
pasta_modelos = Path("..") / "models"
pasta_modelos.mkdir(exist_ok=True)
caminho_modelo = pasta_modelos / "mnist_mlp.joblib"
joblib.dump(modelo_mlp, caminho_modelo)
modelo_recarregado = joblib.load(caminho_modelo)
y_previsto_recarregado=modelo_recarregado.predict(X_test)
print((y_previsto == y_previsto_recarregado).all())


True


## 8. Relatório de métricas

Gravei o relatório em `reports/metrics.json` no formato JSON. Ele registra as métricas do teste (acurácia, F1 ponderado e ROC-AUC), os tamanhos dos conjuntos, os hiperparâmetros do modelo, a versão do scikit-learn (1.9.1) e o contrato de pré-processamento (`normalizacao`: pixels / 255.0, ou seja, dividir os pixels por 255.0).

Os valores vêm das variáveis do notebook e dos atributos do modelo treinado, para que o relatório mostre exatamente o que foi usado.

Este arquivo serve para consultar as métricas do modelo e documentar o contrato de entrada que a API deve seguir (784 valores, divididos por 255.0).

In [8]:
pasta_relatorios = Path("..") / "reports"
pasta_relatorios.mkdir(exist_ok=True)
caminho_relatorio = pasta_relatorios / "metrics.json"

relatorio = {
    "modelo": "MLPClassifier",
    "arquivo_modelo": "models/mnist_mlp.joblib",
    "versao_scikit_learn": sklearn.__version__,
    "acuracia_teste": accuracy,
    "f1_ponderado_teste": f1,
    "roc_auc_ovr_teste": auc,
    "amostras_treino": X_train.shape[0],
    "amostras_validacao": X_val.shape[0],
    "amostras_teste": X_test.shape[0],
    "n_features": modelo_mlp.n_features_in_,
    "normalizacao": "pixels / 255.0",
    "hidden_layer_sizes":modelo_mlp.hidden_layer_sizes,
    "alpha":modelo_mlp.alpha,
    "early_stopping":modelo_mlp.early_stopping,
    "random_state":modelo_mlp.random_state,
    
}

caminho_relatorio.write_text(json.dumps(relatorio, indent=2), encoding="utf-8")
print(caminho_relatorio.read_text(encoding="utf-8"))


{
  "modelo": "MLPClassifier",
  "arquivo_modelo": "models/mnist_mlp.joblib",
  "versao_scikit_learn": "1.9.1",
  "acuracia_teste": 0.978,
  "f1_ponderado_teste": 0.9779868236208817,
  "roc_auc_ovr_teste": 0.9994957849475228,
  "amostras_treino": 49000,
  "amostras_validacao": 7000,
  "amostras_teste": 14000,
  "n_features": 784,
  "normalizacao": "pixels / 255.0",
  "hidden_layer_sizes": [
    128,
    64
  ],
  "alpha": 0.001,
  "early_stopping": true,
  "random_state": 42
}
